In [10]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import joblib

In [11]:
df = pd.read_csv("C:/Users/navya/Downloads/ai-risk-platform/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


Features

In [12]:
X = df.drop("Class", axis=1)
y = df["Class"]

Train Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

Start MLflow Experiment

In [14]:
mlflow.set_experiment("fraud_detection")

<Experiment: artifact_location='file:///C:/Users/navya/mlruns/1', creation_time=1772736506445, experiment_id='1', last_update_time=1772736506445, lifecycle_stage='active', name='fraud_detection', tags={}, workspace='default'>

Train Model with MLflow Tracking

In [15]:
with mlflow.start_run():

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_test)[:,1]

    auc = roc_auc_score(y_test, y_pred)

    mlflow.log_param("model", "xgboost")
    mlflow.log_metric("roc_auc", auc)

    mlflow.sklearn.log_model(model, "fraud_model")

    print("AUC:", auc)

2026/03/05 18:15:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/05 18:15:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC: 0.9615984935054493


In [ ]:
Model

In [16]:
joblib.dump(model, "model.pkl")

['model.pkl']